In [ ]:
# import torch
# from torch.utils.data import DataLoader, random_split, TensorDataset
# import os
# from tqdm import tqdm

# from data_loader.dataset import EuroSatDataset
# from attacks.pgd import PGD
# from config import *

# def create_pgd_test_dataset(
#     model,
#     test_dataset,
#     device,
#     epsilon=0.02,
#     alpha=0.004,
#     iterations=5,
#     batch_size=32,
#     random_start=True
# ):
#     """
#     Create a fixed PGD adversarial test dataset
#     """
#     model.eval()
#     model.to(device)

#     pgd_attack = PGD(
#         model=model,
#         epsilon=epsilon,
#         alpha=alpha,
#         iterations=iterations,
#         random_start=random_start,
#         targeted=False,
#         device=device
#     )

#     test_loader = DataLoader(
#         test_dataset,
#         batch_size=batch_size,
#         shuffle=False
#     )

#     adv_images = []
#     adv_labels = []

#     for images, labels in tqdm(test_loader, desc="Generating PGD test set"):
#         images = images.to(device)
#         labels = labels.to(device)

#         with torch.enable_grad():
#             images_adv = pgd_attack.attack(images, labels)

#         adv_images.append(images_adv.cpu())
#         adv_labels.append(labels.cpu())

#     adv_images = torch.cat(adv_images)
#     adv_labels = torch.cat(adv_labels)

#     adv_test_dataset = TensorDataset(adv_images, adv_labels)

#     return adv_test_dataset



# def generate_and_save_pgd_test(
#     model,
#     test_dataset,
#     device,
#     save_path="datasets"
# ):
#     adv_test_dataset = create_pgd_test_dataset(
#         model=model,
#         test_dataset=test_dataset,
#         device=device,
#         epsilon=0.02,
#         alpha=0.004,
#         iterations=5
#     )

#     torch.save(
#         adv_test_dataset,
#         os.path.join(save_path, "test_pgd_eps002.pt")
#     )

#     print("Saved PGD adversarial test set")


# Clean Train and Test Datasets Creation

In [1]:
import os
import shutil
import random
from pathlib import Path

In [2]:
def create_and_save_datasets(
    data_path,
    save_path,
    train_ratio=0.8,
    seed=42
):
    random.seed(seed)

    data_path = Path(data_path)
    save_path = Path(save_path)

    train_dir = save_path / "train_clean"
    test_dir = save_path / "test_clean"

    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    classes = [d for d in data_path.iterdir() if d.is_dir()]

    print(f"Found {len(classes)} classes")

    for class_dir in classes:
        class_name = class_dir.name

        # Créer les dossiers de sortie
        (train_dir / class_name).mkdir(exist_ok=True)
        (test_dir / class_name).mkdir(exist_ok=True)

        # Lister les images
        images = list(class_dir.glob("*"))
        images = [img for img in images if img.suffix.lower() in [".jpg", ".png", ".jpeg"]]

        random.shuffle(images)

        n_train = int(len(images) * train_ratio)

        train_images = images[:n_train]
        test_images = images[n_train:]

        # Copier les fichiers
        for img_path in train_images:
            shutil.copy(img_path, train_dir / class_name / img_path.name)

        for img_path in test_images:
            shutil.copy(img_path, test_dir / class_name / img_path.name)

        print(
            f"Class {class_name}: "
            f"{len(train_images)} train / {len(test_images)} test"
        )

    print("\nDataset split completed.")
    print(f"Train directory: {train_dir}")
    print(f"Test directory: {test_dir}")


create_and_save_datasets(data_path="data/EuroSAT_RGB", save_path="datasets/EuroSAT_RGB", train_ratio=0.8, seed=42)

Found 10 classes
Class Pasture: 1600 train / 400 test
Class Residential: 2400 train / 600 test
Class River: 2000 train / 500 test
Class Forest: 2400 train / 600 test
Class Industrial: 2000 train / 500 test
Class SeaLake: 2400 train / 600 test
Class Highway: 2000 train / 500 test
Class AnnualCrop: 2400 train / 600 test
Class PermanentCrop: 2000 train / 500 test
Class HerbaceousVegetation: 2400 train / 600 test

Dataset split completed.
Train directory: datasets/EuroSAT_RGB/train_clean
Test directory: datasets/EuroSAT_RGB/test_clean


# Baseline Model Creation

In [ ]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "20",
    "--lr", "0.001",
    "--batch-size", "32",
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/baseline",
    "--save-plots-path", "outputs/plots/baseline_clean",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Training resnet18 for 50 epochs...

Epoch 1/50
--------------------------------------------------


Train Loss: 1.1923 | Train Acc: 57.45%
Val Loss: 1.0645 | Val Acc: 59.98%
Saved best model with val_acc: 59.98%

Epoch 2/50
--------------------------------------------------


Train Loss: 0.8077 | Train Acc: 71.57%
Val Loss: 0.6578 | Val Acc: 76.57%
Saved best model with val_acc: 76.57%

Epoch 3/50
--------------------------------------------------


Train Loss: 0.6824 | Train Acc: 75.91%
Val Loss: 0.5900 | Val Acc: 78.33%
Saved best model with val_acc: 78.33%

Epoch 4/50
--------------------------------------------------


Train Loss: 0.5962 | Train Acc: 78.88%
Val Loss: 0.5258 | Val Acc: 82.64%
Saved best model with val_acc: 82.64%

Epoch 5/50
--------------------------------------------------


...

Train Loss: 0.2961 | Train Acc: 89.85%
Val Loss: 0.2930 | Val Acc: 90.12%
Saved best model with val_acc: 90.12%

Epoch 11/50
--------------------------------------------------


Train Loss: 0.2591 | Train Acc: 91.22%
Val Loss: 0.3821 | Val Acc: 88.07%

Epoch 12/50
--------------------------------------------------


Train Loss: 0.2448 | Train Acc: 91.45%
Val Loss: 0.2333 | Val Acc: 92.05%
Saved best model with val_acc: 92.05%

Epoch 13/50
--------------------------------------------------


Train Loss: 0.2264 | Train Acc: 92.33%
Val Loss: 0.2179 | Val Acc: 92.47%
Saved best model with val_acc: 92.47%

Epoch 14/50
--------------------------------------------------


Train Loss: 0.2150 | Train Acc: 92.78%
Val Loss: 0.2772 | Val Acc: 90.54%

Epoch 15/50
--------------------------------------------------


# Adversarial Test Dataset Creation

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm import tqdm
import json

from models import ResNet18
from data_loader.dataset import EuroSatDataset
from attacks.pgd import PGD
from config import BATCH_SIZE, DEVICE, MEAN, STD, SEED

In [2]:
def load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=5,
    std=STD
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load model
    model = ResNet18().to(device)

    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        print(f"Model loaded from {model_path}")
    else:
        raise FileNotFoundError(f"Model not found at {model_path}")

    model.eval()

    # Load clean test dataset
    dataset = EuroSatDataset(
        root_dir=test_clean_path,
        train=False
    )

    class_names = dataset.classes

    test_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    # Prepare save folders
    os.makedirs(save_adv_path, exist_ok=True)
    for cls in class_names:
        os.makedirs(os.path.join(save_adv_path, cls), exist_ok=True)

    # PGD attack
    epsilon = torch.tensor([epsilon_pixel / s for s in STD]).view(1,3,1,1)
    alpha   = torch.tensor([alpha_pixel / s for s in STD]).view(1,3,1,1)

    pgd = PGD(
        model=model,
        epsilon=epsilon,
        alpha=alpha,
        iterations=iterations,
        random_start=True,
        device=device,
        seed=SEED,
    )

    # Generate & save adversarial images
    img_idx = 0

    for images, labels in tqdm(test_loader, desc="Generating adversarial test set"):
        images = images.to(device)
        labels = labels.to(device)

        with torch.enable_grad():
            adv_images = pgd.attack(images, labels)

        for i in range(adv_images.size(0)):
            label = labels[i].item()
            class_name = class_names[label]

            save_path = os.path.join(
                save_adv_path,
                class_name,
                f"img_{img_idx}.png"
            )

            save_image(adv_images[i], save_path)
            img_idx += 1

    print(f"\nAdversarial test dataset saved to: {save_adv_path}")

    
    attack_config = {
        "attack": "PGD",
        "epsilon": epsilon.tolist() if torch.is_tensor(epsilon) else epsilon,
        "alpha": alpha.tolist() if torch.is_tensor(alpha) else alpha,
        "iterations": iterations,
        "random_start": True,
        "seed": SEED,
        "normalization": {
            "mean": MEAN,
            "std": STD
        }
    }

    os.makedirs("attacks/configs", exist_ok=True)
    with open(os.path.join("attacks/configs", "attack_config.json"), "w") as f:
        json.dump(attack_config, f, indent=4)

    return None

load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/baseline/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=10,
    std=STD
)

Using device: cuda
Model loaded from outputs/models/baseline/best_model.pth
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

         [[0.0179]],

         [[0.0178]]]]) might be too large for ε=tensor([[[[0.0873]],

         [[0.0893]],

         [[0.0889]]]]), iterations=10


Generating adversarial test set: 100%|██████████| 169/169 [03:25<00:00,  1.22s/it]


Adversarial test dataset saved to: datasets/EuroSAT_RGB/test_pgd_eps002


In [1]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/baseline_pgd_eps002",
    "--save-model-path", "outputs/model/baseline",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Evaluating model on test set...
Test Accuracy: 11.20%
Confusion matrix saved to: outputs/plots/baseline_pgd_eps002/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.00      0.00      0.00       600
              Forest       0.00      0.00      0.00       600
HerbaceousVegetatio

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/baseline_pgd_eps002/sample_predictions.png
Batch accuracy on 16 samples: 0.00%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


# Madry

## epsilon = 0.03, alpha = 0.008, pgd_steps = 7

In [1]:
import sys
import json

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "20",
    "--lr", "0.001",
    "--batch-size", "32",
    "--madry", json.dumps({
        "epsilon": 0.03,
        "alpha": 0.008,
        "pgd_steps": 7
    }),
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/madry_eps003",
    "--save-plots-path", "outputs/plots/madry_eps003",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Training resnet18 for 50 epochs...

Epoch 1/50
--------------------------------------------------


Train Loss: 1.9244 | Train Acc: 23.77%
Val Loss: 2.7462 | Val Acc: 19.52%
Saved best model with val_acc: 19.52%

Epoch 2/50
--------------------------------------------------


Train Loss: 1.6813 | Train Acc: 33.97%
Val Loss: 4.1539 | Val Acc: 23.02%
Saved best model with val_acc: 23.02%

Epoch 3/50
--------------------------------------------------


Train Loss: 1.5876 | Train Acc: 37.35%
Val Loss: 2.2232 | Val Acc: 33.01%
Saved best model with val_acc: 33.01%

Epoch 4/50
--------------------------------------------------


Train Loss: 1.5438 | Train Acc: 38.99%
Val Loss: 7.6729 | Val Acc: 12.25%

Epoch 5/50
--------------------------------------------------


...

Train Loss: 1.2522 | Train Acc: 48.90%
Val Loss: 16.8578 | Val Acc: 13.83%

Epoch 20/50
--------------------------------------------------


Train Loss: 1.2359 | Train Acc: 49.66%
Val Loss: 14.9482 | Val Acc: 12.95%

Epoch 21/50
--------------------------------------------------


Train Loss: 1.2372 | Train Acc: 49.75%
Val Loss: 14.6199 | Val Acc: 12.50%

Epoch 22/50
--------------------------------------------------


Train Loss: 1.2294 | Train Acc: 49.57%
Val Loss: 17.6678 | Val Acc: 12.70%

Epoch 23/50
--------------------------------------------------


Train Loss: 1.2308 | Train Acc: 50.08%
Val Loss: 17.1247 | Val Acc: 13.15%

Early stopping triggered after 23 epochs with no improvement.

Training completed! Best validation accuracy: 33.01%
Training history plot saved to: outputs/plots/madry_eps003/resnet18_training_history.png
Training history plot saved to: outputs/plots/madry_eps003/resnet18_training_history.png
Training metrics saved to CSV: outputs/plots/madry_eps003/resnet18_training_metrics.csv
Comprehensive training report saved to: outputs/plots/madry_eps003/resnet18_training_report.txt

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 36.37%
Confusion matrix saved to: outputs/plots/madry_eps003/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.79      0.67      0.72       600
              Forest       0.00      0.00      0.00       600
HerbaceousVegetation       0.12      0.00      0.01       600

In [1]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/madry_eps003/test_pgd_eps002",
    "--save-model-path", "outputs/models/madry_eps003",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 22.74%
Confusion matrix saved to: outputs/plots/madry_eps003/test_pgd_eps002/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.89      0.30      0.45       600
              Forest       0.00      0.

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/madry_eps003/test_pgd_eps002/sample_predictions.png
Batch accuracy on 16 samples: 37.50%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


## epsilon = 0.01, alpha = 0.002, pgd_steps = 7

In [1]:
import sys
import json

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "20",
    "--lr", "0.001",
    "--batch-size", "32",
    "--madry", json.dumps({
        "epsilon": 0.01,
        "alpha": 0.002,
        "pgd_steps": 7
    }),
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/madry_eps001",
    "--save-plots-path", "outputs/plots/madry_eps001",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Training resnet18 for 50 epochs...

Epoch 1/50
--------------------------------------------------


Train Loss: 1.8734 | Train Acc: 24.72%
Val Loss: 93.6347 | Val Acc: 11.73%
Saved best model with val_acc: 11.73%

Epoch 2/50
--------------------------------------------------


Train Loss: 1.6098 | Train Acc: 35.90%
Val Loss: 37.8850 | Val Acc: 15.97%
Saved best model with val_acc: 15.97%

Epoch 3/50
--------------------------------------------------


Train Loss: 1.4783 | Train Acc: 40.16%
Val Loss: 35.6116 | Val Acc: 20.83%
Saved best model with val_acc: 20.83%

Epoch 4/50
--------------------------------------------------


Train Loss: 1.3995 | Train Acc: 43.66%
Val Loss: 18.7316 | Val Acc: 24.18%
Saved best model with val_acc: 24.18%

Epoch 5/50
--------------------------------------------------


...

Train Loss: 0.7946 | Train Acc: 67.15%
Val Loss: 15.1109 | Val Acc: 18.49%

Epoch 38/50
--------------------------------------------------


Train Loss: 0.7960 | Train Acc: 67.22%
Val Loss: 15.1463 | Val Acc: 17.87%

Epoch 39/50
--------------------------------------------------


Train Loss: 0.7946 | Train Acc: 67.12%
Val Loss: 16.0601 | Val Acc: 18.26%

Early stopping triggered after 39 epochs with no improvement.

Training completed! Best validation accuracy: 25.74%
Training history plot saved to: outputs/plots/madry_eps001/resnet18_training_history.png
Training history plot saved to: outputs/plots/madry_eps001/resnet18_training_history.png
Training metrics saved to CSV: outputs/plots/madry_eps001/resnet18_training_metrics.csv
Comprehensive training report saved to: outputs/plots/madry_eps001/resnet18_training_report.txt

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 27.20%
Confusion matrix saved to: outputs/plots/madry_eps001/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.92      0.40      0.56       600
              Forest       0.00      0.00      0.00       600
HerbaceousVegetation       0.77      0.03      0.05       600

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/madry_eps001/sample_predictions.png
Batch accuracy on 16 samples: 31.25%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


In [2]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/madry_eps001/test_pgd_eps002",
    "--save-model-path", "outputs/models/madry_eps001",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 10.44%
Confusion matrix saved to: outputs/plots/madry_eps001/test_pgd_eps002/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.86      0.01      0.02       600
              Forest       0.00      0.

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/madry_eps001/test_pgd_eps002/sample_predictions.png
Batch accuracy on 16 samples: 0.00%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


# Mixed Train Dataset Creation